# Btrick — Final Preliminary-Round Notebook

**Task:** Binary chest X-ray classification — `Normal` vs `Pneumonia`

This notebook documents and reproduces the pipeline actually used by team **Btrick**.

### Organizer rubric coverage

| Criterion | Evidence in this notebook |
|---|---|
| Preprocessing | grayscale, 3-channel replication, square padding, resize 224×224, ImageNet normalization |
| Data augmentation | random affine, translation, scaling, brightness and contrast adjustment — training only |
| CNN / Transfer Learning | ImageNet-pretrained DenseNet121 |
| Result analysis | loss curve, validation accuracy curve, confusion matrix, F1 table, ROC/AUC |
| Pipeline presentation | explicit system pipeline + exact DenseNet121/ensemble specification |

### Final locked system

- DenseNet121 transfer learning
- input tensor: `224 × 224 × 3`
- two training seeds: `2026` and `3407`
- five folds per seed → **10 models**
- seed-group weights: **0.70 / 0.30**
- no TTA
- Pneumonia threshold: **0.576**
- label convention: `Normal = 0`, `Pneumonia = 1`

The locked threshold and ensemble weights are not changed after the sealed internal holdout was opened.


## 1. Imports and portable configuration

The notebook does **not** depend on a private Google Drive path.

For local/Kaggle use, set environment variables or edit the paths in this cell:

- `BTRICK_TRAIN_ROOT`: folder containing `NORMAL/` and `PNEUMONIA/`
- `BTRICK_TEST_DIR`: organizer-provided unlabeled test folder
- `BTRICK_MODEL_PATH`: path to `Btrick.keras`

When the model is already supplied, training can remain disabled and the final inference section can be run directly.


In [ ]:
import os, sys, json, random, hashlib, gc, time, pickle, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFile

from sklearn.metrics import (
    confusion_matrix, f1_score, roc_auc_score, roc_curve
)

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

import tensorflow as tf

ImageFile.LOAD_TRUNCATED_IMAGES=False

TEAM_NAME='Btrick'
LABEL_TO_ID={'Normal':0,'Pneumonia':1}
ID_TO_LABEL=np.array(['Normal','Pneumonia'])

IMAGE_SIZE=224
N_FOLDS=5
HOLDOUT_FRACTION=0.15
SEEDS=[2026,3407]
SEED_WEIGHTS={2026:0.70,3407:0.30}
THRESHOLD=0.576
TTA=False

BATCH_SIZE=8
GRAD_ACCUM=2
EPOCHS=8
PATIENCE=3
LR=1e-4
WEIGHT_DECAY=1e-4

IMAGENET_MEAN=np.array([0.485,0.456,0.406],dtype=np.float32)
IMAGENET_STD=np.array([0.229,0.224,0.225],dtype=np.float32)

VALID_EXTENSIONS={'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}

def find_project_root():
    candidates=[Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p/'NORMAL').is_dir() and (p/'PNEUMONIA').is_dir():
            return p
        if (p/'cv_outputs').is_dir() and (p/'notebooks').is_dir():
            return p
    return Path.cwd()

PROJECT_ROOT=find_project_root()
TRAIN_ROOT=Path(os.environ.get('BTRICK_TRAIN_ROOT',str(PROJECT_ROOT)))
TEST_DIR_ENV=os.environ.get('BTRICK_TEST_DIR')
TEST_DIR=Path(TEST_DIR_ENV) if TEST_DIR_ENV else None

OUTPUT_DIR=PROJECT_ROOT/'final_submission'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

MODEL_CANDIDATES=[
    Path(os.environ['BTRICK_MODEL_PATH']) if 'BTRICK_MODEL_PATH' in os.environ else None,
    PROJECT_ROOT/'final_submission'/'Btrick.keras',
    PROJECT_ROOT/'Btrick.keras',
    PROJECT_ROOT/'conversion_outputs'/'full_keras_ensemble'/'locked_densenet10_ensemble_core.keras',
]
MODEL_CANDIDATES=[p for p in MODEL_CANDIDATES if p is not None]

RUN_TRAINING=False  # Set True only to reproduce the expensive 2-seed × 5-fold training stage.

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'

print('Python:',sys.version.split()[0])
print('PyTorch:',torch.__version__)
print('TensorFlow:',tf.__version__)
print('Training device:',DEVICE)
print('Project root:',PROJECT_ROOT)
print('Train root:',TRAIN_ROOT)
print('Final threshold:',THRESHOLD)
print('Run training:',RUN_TRAINING)


## 2. Data loading, labels and integrity audit

The training dataset is expected to contain two class folders:

```text
TRAIN_ROOT/
├── NORMAL/
└── PNEUMONIA/
```

The audit computes SHA-256 hashes so exact duplicates can be kept in the same holdout/fold and cannot leak across validation boundaries.


In [ ]:
def image_files(root):
    root=Path(root)
    return sorted(
        [
            p for p in root.rglob('*')
            if p.is_file()
            and p.suffix.lower() in VALID_EXTENSIONS
            and not p.name.startswith('._')   # ignore macOS AppleDouble metadata files
            and not p.name.startswith('.')    # ignore other hidden dot-files
        ],
        key=lambda p:str(p).lower()
    )

def sha256_file(path,chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def discover_training_data(train_root):
    train_root=Path(train_root)
    class_dirs={
        'Normal': train_root/'NORMAL',
        'Pneumonia': train_root/'PNEUMONIA',
    }
    rows=[]
    for label,d in class_dirs.items():
        if not d.is_dir():
            raise FileNotFoundError(f'Missing class folder: {d}')
        for p in image_files(d):
            try:
                with Image.open(p) as im:
                    im.verify()
                ok=True
            except Exception:
                ok=False
            rows.append({
                'label':label,
                'filename':p.name,
                'path':str(p),
                'read_ok':ok,
                'sha256':sha256_file(p) if ok else None,
            })
    frame=pd.DataFrame(rows)
    if not frame['read_ok'].all():
        bad=frame.loc[~frame.read_ok,'path'].tolist()
        raise RuntimeError(f'Unreadable images found: {bad[:5]}')
    cross=frame.groupby('sha256')['label'].nunique()
    if (cross>1).any():
        raise RuntimeError('Cross-label exact duplicate group detected.')
    return frame

# Local convenience: reuse the audited manifest if this project directory is present.
AUDIT_CSV=PROJECT_ROOT/'audit_outputs'/'image_audit.csv'
if AUDIT_CSV.exists():
    audit_df=pd.read_csv(AUDIT_CSV)
    audit_df=audit_df[['label','filename','path','read_ok','sha256']].copy()
    print('Loaded existing audit:',AUDIT_CSV)
elif (TRAIN_ROOT/'NORMAL').is_dir() and (TRAIN_ROOT/'PNEUMONIA').is_dir():
    audit_df=discover_training_data(TRAIN_ROOT)
    print('Created audit from TRAIN_ROOT')
else:
    audit_df=None
    print('Training data not present in this runtime; training/audit cells can be skipped for inference-only use.')

if audit_df is not None:
    display(audit_df['label'].value_counts().reindex(['Normal','Pneumonia']).to_frame('images'))
    print('Total:',len(audit_df))
    print('Exact duplicate groups:',int((audit_df.groupby('sha256').size()>1).sum()))


## 3. Leakage-resistant split protocol

The original development protocol used:

- **15% sealed internal holdout**
- remaining **85% development data**
- **5 duplicate-aware folds** inside development data
- exact duplicate images (same SHA-256) always remain together

This prevents the same image bytes from appearing in both training and validation/holdout.


In [ ]:
def make_split_manifest(df,seed=2026,holdout_fraction=0.15,n_folds=5):
    df=df.copy().reset_index(drop=True)

    groups=(df.groupby(['sha256','label'],as_index=False)
              .agg(group_size=('filename','size')))

    rng=np.random.default_rng(seed)
    holdout_hashes=set()

    for label in ['Normal','Pneumonia']:
        g=groups[groups.label==label].copy()
        g=g.iloc[rng.permutation(len(g))].reset_index(drop=True)
        target=int(round((df.label==label).sum()*holdout_fraction))

        running=0
        selected=[]
        for _,row in g.iterrows():
            if running>=target:
                break
            selected.append(row.sha256)
            running+=int(row.group_size)
        holdout_hashes.update(selected)

    df['split']=np.where(df.sha256.isin(holdout_hashes),'holdout','dev')
    df['cv_fold']=-1

    dev_groups=groups[~groups.sha256.isin(holdout_hashes)].copy()
    fold_counts={label:[0]*n_folds for label in ['Normal','Pneumonia']}

    for label in ['Normal','Pneumonia']:
        g=dev_groups[dev_groups.label==label].copy()
        g=g.iloc[rng.permutation(len(g))].copy()
        g=g.sort_values('group_size',ascending=False,kind='stable')
        assignment={}

        for _,row in g.iterrows():
            minimum=min(fold_counts[label])
            candidates=[i for i,c in enumerate(fold_counts[label]) if c==minimum]
            fold=int(rng.choice(candidates))
            assignment[row.sha256]=fold
            fold_counts[label][fold]+=int(row.group_size)

        mask=(df.split=='dev') & (df.label==label)
        df.loc[mask,'cv_fold']=df.loc[mask,'sha256'].map(assignment).astype(int)

    assert set(df.loc[df.split=='holdout','cv_fold'])=={-1}
    assert (df.loc[df.split=='dev','cv_fold']>=0).all()

    # Exact duplicate leakage gate.
    leak=df.groupby('sha256').agg(
        split_n=('split','nunique'),
        fold_n=('cv_fold','nunique')
    )
    assert (leak.split_n==1).all()

    return df

MANIFEST_CSV=PROJECT_ROOT/'split_outputs'/'split_manifest.csv'

if MANIFEST_CSV.exists():
    manifest=pd.read_csv(MANIFEST_CSV)
    print('Loaded frozen split manifest:',MANIFEST_CSV)
elif audit_df is not None:
    manifest=make_split_manifest(audit_df)
    print('Recreated deterministic split manifest.')
else:
    manifest=None

if manifest is not None:
    display(pd.crosstab(manifest.split,manifest.label))
    display(pd.crosstab(
        manifest.loc[manifest.split=='dev','cv_fold'],
        manifest.loc[manifest.split=='dev','label']
    ))


## 4. Preprocessing and data augmentation

### Preprocessing used for training, validation and TEST

```text
Input X-ray
→ convert to grayscale
→ replicate grayscale to 3 channels
→ pad to square (black)
→ resize to 224 × 224
→ convert to float [0,1]
→ ImageNet normalization
```

### Training-only augmentation

```text
RandomAffine:
  rotation ±5°
  translation ±2%
  scale 0.95–1.05
+
ColorJitter:
  brightness 0.08
  contrast 0.08
```

**No augmentation and no TTA are used on validation/TEST images.**


In [ ]:
class PadToSquare:
    def __call__(self,img):
        w,h=img.size
        s=max(w,h)
        left=(s-w)//2
        right=s-w-left
        top=(s-h)//2
        bottom=s-h-top
        return ImageOps.expand(img,border=(left,top,right,bottom),fill=0)

pad_square=PadToSquare()

base_image_transform=transforms.Compose([
    PadToSquare(),
    transforms.Resize(
        (IMAGE_SIZE,IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
])

train_tensor_transform=transforms.Compose([
    transforms.RandomAffine(
        degrees=5,
        translate=(0.02,0.02),
        scale=(0.95,1.05),
        fill=0
    ),
    transforms.ColorJitter(brightness=0.08,contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(),IMAGENET_STD.tolist())
])

eval_tensor_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(),IMAGENET_STD.tolist())
])

class XrayDataset(Dataset):
    def __init__(self,frame,training=False):
        self.df=frame.reset_index(drop=True)
        self.training=training

    def __len__(self):
        return len(self.df)

    def __getitem__(self,i):
        row=self.df.iloc[i]
        with Image.open(row.path) as im:
            im=im.convert('L').convert('RGB')
            im=base_image_transform(im)

        tfm=train_tensor_transform if self.training else eval_tensor_transform
        x=tfm(im)
        y=LABEL_TO_ID[row.label]
        return x,y,row.filename

print('Preprocessing and augmentation definitions ready.')


## 5. DenseNet121 transfer-learning architecture

Backbone: **DenseNet121 pretrained on ImageNet**.

Main feature stages:

```text
224×224×3
→ Conv 7×7 / stride 2, 64 channels
→ MaxPool 3×3 / stride 2
→ Dense Block 1: 6 layers, growth rate 32
→ Transition 1
→ Dense Block 2: 12 layers
→ Transition 2
→ Dense Block 3: 24 layers
→ Transition 3
→ Dense Block 4: 16 layers
→ BatchNorm + ReLU
→ Global Average Pooling
→ Fully Connected: 1024 → 2
→ Softmax probabilities
```

The final Btrick system contains **10 trained DenseNet121 models**:
5 folds for seed 2026 + 5 folds for seed 3407.


In [ ]:
def make_densenet121(pretrained=True):
    weights=models.DenseNet121_Weights.DEFAULT if pretrained else None
    model=models.densenet121(weights=weights)
    model.classifier=nn.Linear(model.classifier.in_features,2)
    return model.to(DEVICE)

model_example=make_densenet121(pretrained=False)
print(model_example)
print('Classifier:',model_example.classifier)
del model_example
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6. Competition metric and training procedure

Competition score:

[
P = 0.30\,Accuracy + 0.25\,Sensitivity + 0.20\,Specificity + 0.25\,F1
]

Training configuration used by Btrick:

- weighted cross-entropy
- AdamW, learning rate `1e-4`
- weight decay `1e-4`
- cosine annealing
- 8 epochs maximum
- early stopping patience = 3
- physical batch size = 8
- gradient accumulation = 2 → effective batch = 16
- automatic mixed precision on CUDA


In [ ]:
def competition_metrics(y_true,p_pneumonia,threshold=0.5):
    y=np.asarray(y_true,dtype=int)
    p=np.asarray(p_pneumonia,dtype=float)
    pred=(p>=threshold).astype(int)

    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    accuracy=(tp+tn)/len(y)
    sensitivity=tp/max(tp+fn,1)
    specificity=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    auc=roc_auc_score(y,p)

    score=.30*accuracy+.25*sensitivity+.20*specificity+.25*f1

    return {
        'P':float(score),
        'Accuracy':float(accuracy),
        'Sensitivity':float(sensitivity),
        'Specificity':float(specificity),
        'F1':float(f1),
        'AUROC':float(auc),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        'threshold':float(threshold),
    }

@torch.no_grad()
def predict_torch(model,loader):
    model.eval()
    ys,ps,names=[],[],[]

    for x,y,n in loader:
        x=x.to(DEVICE)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        probs=torch.softmax(logits.float(),dim=1)[:,1]

        ys.extend(y.numpy())
        ps.extend(probs.cpu().numpy())
        names.extend(list(n))

    return np.asarray(ys),np.asarray(ps),names


In [ ]:
def train_one_fold(train_df,val_df,seed,fold,output_dir):
    random.seed(seed+fold)
    np.random.seed(seed+fold)
    torch.manual_seed(seed+fold)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed+fold)

    train_loader=DataLoader(
        XrayDataset(train_df,training=True),
        batch_size=BATCH_SIZE,shuffle=True,num_workers=0
    )
    val_loader=DataLoader(
        XrayDataset(val_df,training=False),
        batch_size=BATCH_SIZE,shuffle=False,num_workers=0
    )

    model=make_densenet121(pretrained=True)

    counts=train_df.label.value_counts()
    n0=int(counts.Normal); n1=int(counts.Pneumonia)
    total=n0+n1
    class_weights=torch.tensor(
        [total/(2*n0),total/(2*n1)],
        dtype=torch.float32,device=DEVICE
    )

    criterion=nn.CrossEntropyLoss(weight=class_weights)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best_score=-np.inf
    best_state=None
    best_epoch=None
    stale=0
    history=[]

    for epoch in range(1,EPOCHS+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        loss_sum=0.0
        seen=0

        for batch_idx,(x,y,_) in enumerate(train_loader,1):
            x=x.to(DEVICE)
            y=y.to(DEVICE)

            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                logits=model(x)
                raw_loss=criterion(logits,y)
                loss=raw_loss/GRAD_ACCUM

            scaler.scale(loss).backward()

            if batch_idx%GRAD_ACCUM==0 or batch_idx==len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            loss_sum+=float(raw_loss.item())*len(x)
            seen+=len(x)

        scheduler.step()

        y_val,p_val,_=predict_torch(model,val_loader)
        metrics=competition_metrics(y_val,p_val,0.5)

        history.append({
            'seed':seed,
            'fold':fold,
            'epoch':epoch,
            'train_loss':loss_sum/seen,
            **metrics
        })

        print(
            f"seed={seed} fold={fold} epoch={epoch} "
            f"loss={loss_sum/seen:.4f} "
            f"P={metrics['P']:.5f} "
            f"Acc={metrics['Accuracy']:.5f} "
            f"F1={metrics['F1']:.5f}"
        )

        if metrics['P']>best_score+1e-6:
            best_score=metrics['P']
            best_epoch=epoch
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            stale=0
        else:
            stale+=1
            if stale>=PATIENCE:
                break

    output_dir=Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)

    checkpoint={
        'architecture':'densenet121',
        'preprocessing':'force_grayscale_then_rgb',
        'seed':seed,
        'fold':fold,
        'best_epoch':best_epoch,
        'state_dict':best_state,
    }

    torch.save(checkpoint,output_dir/f'fold{fold}_best.pt')
    pd.DataFrame(history).to_csv(output_dir/f'fold{fold}_history.csv',index=False)

    return history


### Optional full retraining

The following cell is intentionally disabled by default because it trains **10 DenseNet121 models**.

Set `RUN_TRAINING=True` only when reproducing the training stage from the organizer-provided training set.


In [ ]:
if RUN_TRAINING:
    if manifest is None:
        raise RuntimeError('Training data/split manifest is required for retraining.')

    dev=manifest[manifest.split=='dev'].copy()

    for seed in SEEDS:
        seed_dir=OUTPUT_DIR/f'training_seed{seed}'
        for fold in range(N_FOLDS):
            train_df=dev[dev.cv_fold!=fold].copy()
            val_df=dev[dev.cv_fold==fold].copy()

            print('\n'+'='*80)
            print('TRAIN',seed,'FOLD',fold)
            print('='*80)

            train_one_fold(train_df,val_df,seed,fold,seed_dir)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
else:
    print('RUN_TRAINING=False — using the already trained and supplied Btrick.keras for final inference.')


## 7. Locked ensemble and internal evaluation

The model-selection pipeline was frozen **before** the sealed internal holdout was opened.

### Locked 5-fold OOF performance

- P = **0.9869985311**
- Accuracy = **0.9860155892**
- Sensitivity = **0.9864947821**
- Specificity = **0.9846014493**
- F1 = **0.9905994760**
- AUROC = **0.9981606036**
- Confusion matrix: TN=1087, FP=17, FN=44, TP=3214

### One-time sealed holdout performance

- 770 images: 195 Normal, 575 Pneumonia
- P = **0.9796492012**
- Accuracy = **0.9805194805**
- Sensitivity = **0.9878260870**
- Specificity = **0.9589743590**
- F1 = **0.9869678540**
- AUROC = **0.9953444816**
- Confusion matrix: TN=187, FP=8, FN=7, TP=568

The holdout result was **not used to retune** the threshold, architecture, or ensemble weights.


In [ ]:
LOCKED_OOF_METRICS={
    'P':0.9869985311366707,
    'Accuracy':0.9860155892,
    'Sensitivity':0.9864947821,
    'Specificity':0.9846014493,
    'F1':0.9905994760,
    'AUROC':0.9981606036,
    'TN':1087,'FP':17,'FN':44,'TP':3214,
    'threshold':0.576,
}

SEALED_HOLDOUT_METRICS={
    'P':0.9796492012,
    'Accuracy':0.9805194805,
    'Sensitivity':0.9878260870,
    'Specificity':0.9589743590,
    'F1':0.9869678540,
    'AUROC':0.9953444816,
    'TN':187,'FP':8,'FN':7,'TP':568,
    'threshold':0.576,
}

metrics_table=pd.DataFrame([
    {'Evaluation':'5-fold OOF',**LOCKED_OOF_METRICS},
    {'Evaluation':'Sealed internal holdout',**SEALED_HOLDOUT_METRICS},
])

display(metrics_table[[
    'Evaluation','P','Accuracy','Sensitivity','Specificity','F1','AUROC'
]])


## 8. Result-analysis figures required by the rubric

For the submitted notebook, these cells read the original training histories and sealed-holdout predictions when those project artifacts are available.

If the complete pipeline is retrained with `RUN_TRAINING=True`, the same plots can be regenerated from the new histories and predictions.

The Accuracy Curve below is the **validation accuracy by epoch**. Training accuracy was not recorded in the original training logs, so it is not fabricated.


In [ ]:
# Load original 10 training histories when running inside the Btrick project.
history_dirs=[
    PROJECT_ROOT/'cv_outputs'/'densenet121_224_gray_b8a2',
    PROJECT_ROOT/'cv_outputs'/'densenet121_224_gray_b8a2_seed3407',
]

history_frames=[]
for d in history_dirs:
    if d.exists():
        for fold in range(5):
            p=d/f'fold{fold}_history.csv'
            if p.exists():
                h=pd.read_csv(p)
                h['source_dir']=d.name
                h['fold_id']=fold
                history_frames.append(h)

if history_frames:
    history_df=pd.concat(history_frames,ignore_index=True)

    plt.figure(figsize=(8,4.5))
    for _,g in history_df.groupby(['source_dir','fold_id']):
        g=g.sort_values('epoch')
        plt.plot(g.epoch,g.train_loss,alpha=.25)
    mean_curve=history_df.groupby('epoch',as_index=False).train_loss.mean()
    plt.plot(mean_curve.epoch,mean_curve.train_loss,linewidth=2.5,label='Mean across runs')
    plt.xlabel('Epoch'); plt.ylabel('Training loss')
    plt.title('Loss Curve')
    plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,4.5))
    for _,g in history_df.groupby(['source_dir','fold_id']):
        g=g.sort_values('epoch')
        plt.plot(g.epoch,g.Accuracy,alpha=.25)
    mean_curve=history_df.groupby('epoch',as_index=False).Accuracy.mean()
    plt.plot(mean_curve.epoch,mean_curve.Accuracy,linewidth=2.5,label='Mean validation accuracy')
    plt.xlabel('Epoch'); plt.ylabel('Validation accuracy')
    plt.title('Accuracy Curve')
    plt.ylim(.90,1.0)
    plt.grid(alpha=.25); plt.legend(); plt.tight_layout()
    plt.show()
else:
    print('Training-history CSVs are not present in this runtime. The code above is the exact plotting procedure used for the report.')


In [ ]:
holdout_pred_candidates=[
    PROJECT_ROOT/'holdout_outputs'/'locked_pipeline_v1'/'holdout_predictions.csv',
    OUTPUT_DIR/'holdout_predictions.csv',
]
holdout_pred_path=next((p for p in holdout_pred_candidates if p.exists()),None)

if holdout_pred_path is not None:
    hp=pd.read_csv(holdout_pred_path)
    y=hp.y_true.astype(int).to_numpy()
    p=hp.p_locked_ensemble.astype(float).to_numpy()
    pred=hp.prediction.astype(int).to_numpy()

    cm=confusion_matrix(y,pred,labels=[0,1])

    plt.figure(figsize=(5,4.5))
    plt.imshow(cm)
    plt.xticks([0,1],['Normal','Pneumonia'])
    plt.yticks([0,1],['Normal','Pneumonia'])
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.title('Confusion Matrix — Sealed Holdout')
    for i in range(2):
        for j in range(2):
            plt.text(j,i,str(cm[i,j]),ha='center',va='center')
    plt.colorbar(); plt.tight_layout(); plt.show()

    fpr,tpr,_=roc_curve(y,p)
    auc=roc_auc_score(y,p)

    plt.figure(figsize=(5.5,4.5))
    plt.plot(fpr,tpr,linewidth=2,label=f'AUC = {auc:.6f}')
    plt.plot([0,1],[0,1],'--',linewidth=1)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC/AUC Curve — Sealed Holdout')
    plt.grid(alpha=.25); plt.legend(loc='lower right')
    plt.tight_layout(); plt.show()
else:
    print('Sealed-holdout prediction CSV is not present in this runtime.')
    print('Recorded confusion matrix: [[187, 8], [7, 568]]')
    print('Recorded AUROC:',SEALED_HOLDOUT_METRICS['AUROC'])


In [ ]:
f1_table=pd.DataFrame([
    {
        'Evaluation':'5-fold OOF locked ensemble',
        'F1-score':LOCKED_OOF_METRICS['F1'],
        'Accuracy':LOCKED_OOF_METRICS['Accuracy'],
        'Sensitivity':LOCKED_OOF_METRICS['Sensitivity'],
        'Specificity':LOCKED_OOF_METRICS['Specificity'],
        'ROC-AUC':LOCKED_OOF_METRICS['AUROC'],
    },
    {
        'Evaluation':'One-time sealed internal holdout',
        'F1-score':SEALED_HOLDOUT_METRICS['F1'],
        'Accuracy':SEALED_HOLDOUT_METRICS['Accuracy'],
        'Sensitivity':SEALED_HOLDOUT_METRICS['Sensitivity'],
        'Specificity':SEALED_HOLDOUT_METRICS['Specificity'],
        'ROC-AUC':SEALED_HOLDOUT_METRICS['AUROC'],
    }
])

display(f1_table)


## 9. System block pipeline

The system diagram used in the technical report must match this code:

```text
Chest X-ray
    ↓
Convert to grayscale
    ↓
Replicate grayscale to RGB (3 channels)
    ↓
Pad to square
    ↓
Resize 224 × 224
    ↓
ImageNet normalization
    ↓
┌───────────────────────────────────────────────┐
│ TRAIN ONLY:                                  │
│ RandomAffine + translation + scale           │
│ Brightness / contrast jitter                 │
└───────────────────────────────────────────────┘
    ↓
DenseNet121 transfer learning
    ↓
2 seeds × 5 folds = 10 models
    ↓
5-fold average inside each seed
    ↓
0.70 × Seed2026 + 0.30 × Seed3407
    ↓
P(Pneumonia)
    ↓
threshold = 0.576
    ↓
Normal / Pneumonia
```


## 10. Final Keras model

The original PyTorch ensemble was converted to a single standard-layer Keras model and checked for numerical parity.

Conversion validation:

- all **10/10 checkpoints passed**
- maximum full-ensemble probability difference vs PyTorch: **4.47 × 10⁻⁸**
- class agreement at threshold 0.576: **100%**
- Keras save → reload difference: **0**
- final core model size: approximately **285.68 MiB**

The supplied `Btrick.keras` expects an already-preprocessed float32 tensor:

```text
(None, 224, 224, 3)
```

Output:

```text
[P(Normal), P(Pneumonia)]
```


In [ ]:
def locate_final_model():
    for p in MODEL_CANDIDATES:
        if p.exists():
            return p
    raise FileNotFoundError(
        'Btrick.keras not found. Put Btrick.keras beside this notebook, '
        'or set environment variable BTRICK_MODEL_PATH.'
    )

MODEL_PATH=locate_final_model()
print('Loading:',MODEL_PATH)

keras_model=tf.keras.models.load_model(MODEL_PATH,compile=False)

print('Model name:',keras_model.name)
print('Input shape:',keras_model.input_shape)
print('Output shape:',keras_model.output_shape)

assert tuple(keras_model.input_shape[1:])==(224,224,3)
assert tuple(keras_model.output_shape[1:])==(2,)


## 11. Official TEST preprocessing

BTC confirmed that teams may preprocess the downloaded TEST images themselves before passing them to the loaded model.

This function therefore performs the **same validation/test preprocessing used during development** and outputs exactly the model's `224×224×3` float32 input.


In [ ]:
def preprocess_for_keras(path):
    with Image.open(path) as im:
        im=im.convert('L').convert('RGB')

        w,h=im.size
        s=max(w,h)
        left=(s-w)//2
        right=s-w-left
        top=(s-h)//2
        bottom=s-h-top

        im=ImageOps.expand(
            im,
            border=(left,top,right,bottom),
            fill=0
        )

        im=im.resize(
            (IMAGE_SIZE,IMAGE_SIZE),
            Image.Resampling.BILINEAR
        )

        x=np.asarray(im,dtype=np.float32)/255.0

    x=(x-IMAGENET_MEAN)/IMAGENET_STD
    return x.astype(np.float32)

# Shape sanity check when at least one training image is available.
if audit_df is not None and len(audit_df):
    sample=preprocess_for_keras(audit_df.iloc[0].path)
    print('Preprocessed shape:',sample.shape)
    print('dtype:',sample.dtype)
    assert sample.shape==(224,224,3)


## 12. Official TEST inference and evaluation

Ngày thi:

1. Tải thư mục TEST ảnh RAW từ BTC.
2. Đặt `TEST_DIR` hoặc biến môi trường `BTRICK_TEST_DIR`.
3. Notebook tự đọc toàn bộ ảnh hợp lệ, bỏ qua file metadata ẩn như `._*.jpeg`.
4. Ảnh được tiền xử lý đúng pipeline đã khóa rồi đưa vào `Btrick.keras`.
5. Notebook luôn xuất dự đoán và xác suất.
6. Nếu ground truth được BTC cung cấp, notebook tự tìm nhãn và in:
   - Accuracy
   - Sensitivity
   - Specificity
   - F1-score
   - Confusion Matrix
   - ROC/AUC
   - Competition score P

### Các dạng ground truth được hỗ trợ tự động

- Cấu trúc thư mục `Normal/` và `Pneumonia/`, hoặc
- file CSV gần thư mục TEST có cột tên ảnh + cột nhãn.

Nếu TEST chỉ gồm ảnh RAW **không có nhãn thật**, các chỉ số Accuracy/Sensitivity/Specificity/F1/ROC-AUC không thể được tính một cách hợp lệ; notebook sẽ chỉ xuất prediction/probability và báo rõ ground truth chưa có.


In [ ]:
# ===== EDIT THIS ON COMPETITION DAY ONLY IF AUTO-DETECTION DOES NOT FIND THE TEST FOLDER =====
# TEST_DIR = Path(r'/kaggle/input/.../test')
# =============================================================================================

def discover_test_dir():
    if TEST_DIR is not None:
        return Path(TEST_DIR)

    candidates=[
        PROJECT_ROOT/'test',
        PROJECT_ROOT/'Test',
        PROJECT_ROOT/'TEST',
        Path.cwd()/'test',
        Path.cwd()/'Test',
    ]

    valid=[]
    for d in candidates:
        if d.is_dir() and image_files(d):
            valid.append(d)

    unique=[]
    seen=set()
    for d in valid:
        key=str(d.resolve())
        if key not in seen:
            seen.add(key)
            unique.append(d)

    if len(unique)==1:
        return unique[0]

    if len(unique)==0:
        return None

    raise RuntimeError(f'Multiple possible TEST folders found: {unique}. Set TEST_DIR explicitly.')

official_test_dir=discover_test_dir()

if official_test_dir is None:
    print('TEST folder not supplied yet. This is expected before competition day.')
else:
    print('Official TEST folder:',official_test_dir)


In [ ]:
def predict_official_test(model,test_dir,batch_size=1):
    paths=image_files(test_dir)
    if not paths:
        raise RuntimeError(f'No test images found in {test_dir}')

    names=[p.name for p in paths]
    if len(names)!=len(set(names)):
        raise RuntimeError('Duplicate base filenames found in TEST; Image_id would be ambiguous.')

    probabilities=[]

    for start in range(0,len(paths),batch_size):
        batch_paths=paths[start:start+batch_size]
        xb=np.stack([preprocess_for_keras(p) for p in batch_paths],axis=0)

        probs=model(xb,training=False).numpy()
        probabilities.append(probs)

        if (start+len(batch_paths))%50==0 or start+len(batch_paths)==len(paths):
            print(f'Processed {start+len(batch_paths)}/{len(paths)}')

    probs=np.concatenate(probabilities,axis=0)

    assert probs.shape==(len(paths),2)
    assert np.isfinite(probs).all()

    p_pneumonia=probs[:,1]
    pred_idx=(p_pneumonia>=THRESHOLD).astype(int)
    labels=ID_TO_LABEL[pred_idx]

    submission=pd.DataFrame({
        'Image_id':names,
        'Prediction':labels,
    })

    audit=pd.DataFrame({
        'Image_id':names,
        'P_Normal':probs[:,0],
        'P_Pneumonia':p_pneumonia,
        'Prediction':labels,
    })

    return submission,audit

if official_test_dir is not None:
    submission,test_audit=predict_official_test(
        keras_model,
        official_test_dir,
        batch_size=1
    )

    # The organizer's formal example uses this two-column format.
    CSV_PATH=OUTPUT_DIR/'team01.csv'
    submission.to_csv(CSV_PATH,index=False)

    TEST_AUDIT_PATH=OUTPUT_DIR/'Btrick_test_probability_audit.csv'
    test_audit.to_csv(TEST_AUDIT_PATH,index=False)

    assert submission.columns.tolist()==['Image_id','Prediction']
    assert submission.Image_id.is_unique
    assert set(submission.Prediction).issubset({'Normal','Pneumonia'})

    print('Saved:',CSV_PATH)
    print('Prediction counts:',submission.Prediction.value_counts().to_dict())
    display(submission.head(10))
else:
    print('Inference skipped because the organizer TEST folder has not been provided yet.')


## 12.1 Automatic TEST metrics requested by the organizer

This cell evaluates the TEST set **only when true labels are available**. It never treats model predictions as ground truth.


In [ ]:
def _normalize_binary_label(value):
    if pd.isna(value):
        return None
    if isinstance(value,(int,np.integer)):
        return int(value) if int(value) in (0,1) else None
    if isinstance(value,float) and value.is_integer() and int(value) in (0,1):
        return int(value)

    s=str(value).strip().lower()
    mapping={
        '0':0,'normal':0,'negative':0,'neg':0,
        '1':1,'pneumonia':1,'positive':1,'pos':1,
    }
    return mapping.get(s)


def discover_test_ground_truth(test_dir, image_paths_list):
    """
    Return DataFrame(Image_id, y_true, GroundTruth) or None.

    Priority:
      1) class folders Normal/ and Pneumonia/
      2) CSV near TEST_DIR with image-id + label columns
    """
    test_dir=Path(test_dir)
    names=[p.name for p in image_paths_list]

    # ---- 1. Labels encoded in folder names ----
    folder_rows=[]
    folder_ok=True
    for p in image_paths_list:
        try:
            rel=p.relative_to(test_dir)
            parent_parts=[x.lower() for x in rel.parts[:-1]]
        except Exception:
            parent_parts=[]

        label=None
        if 'normal' in parent_parts:
            label=0
        elif 'pneumonia' in parent_parts:
            label=1
        else:
            folder_ok=False
            break

        folder_rows.append({
            'Image_id':p.name,
            'y_true':label,
            'GroundTruth':ID_TO_LABEL[label],
        })

    if folder_ok and len(folder_rows)==len(image_paths_list):
        gt=pd.DataFrame(folder_rows)
        if gt.Image_id.is_unique:
            print('Ground truth source: class-folder names (Normal / Pneumonia)')
            return gt

    # ---- 2. Labels supplied in CSV ----
    search_dirs=[test_dir,test_dir.parent]
    csv_candidates=[]
    seen=set()
    for d in search_dirs:
        if not d.exists():
            continue
        for p in sorted(d.glob('*.csv')):
            key=str(p.resolve()).lower()
            if key not in seen:
                seen.add(key)
                csv_candidates.append(p)

    id_aliases={'image_id','imageid','filename','file_name','file','image','name'}
    label_aliases={'label','groundtruth','ground_truth','class','target','y_true','prediction'}

    for csv_path in csv_candidates:
        try:
            df=pd.read_csv(csv_path)
        except Exception:
            continue

        lower={str(c).strip().lower():c for c in df.columns}
        id_col=next((lower[a] for a in id_aliases if a in lower),None)
        label_col=next((lower[a] for a in label_aliases if a in lower),None)

        if id_col is None or label_col is None:
            continue

        temp=df[[id_col,label_col]].copy()
        temp.columns=['Image_id','raw_label']
        temp['Image_id']=temp['Image_id'].astype(str).map(lambda x:Path(x).name)
        temp['y_true']=temp['raw_label'].map(_normalize_binary_label)
        temp=temp.dropna(subset=['y_true']).copy()
        temp['y_true']=temp['y_true'].astype(int)
        temp=temp[temp.Image_id.isin(names)]

        if len(temp)==len(names) and temp.Image_id.nunique()==len(names):
            temp['GroundTruth']=ID_TO_LABEL[temp.y_true.to_numpy()]
            print('Ground truth source:',csv_path)
            return temp[['Image_id','y_true','GroundTruth']]

    return None


def evaluate_official_test_predictions(test_dir, submission, test_audit):
    paths=image_files(test_dir)
    gt=discover_test_ground_truth(test_dir,paths)

    if gt is None:
        print('\n'+'='*72)
        print('GROUND TRUTH NOT FOUND')
        print('Predictions were generated successfully, but Accuracy, Sensitivity,')
        print('Specificity, F1, Confusion Matrix and ROC/AUC require true TEST labels.')
        print('='*72)
        return None

    merged=test_audit.merge(gt,on='Image_id',how='inner',validate='one_to_one')
    if len(merged)!=len(test_audit):
        raise RuntimeError(
            f'Ground-truth alignment mismatch: predictions={len(test_audit)}, matched={len(merged)}'
        )

    y_true=merged.y_true.to_numpy(dtype=int)
    p=merged.P_Pneumonia.to_numpy(dtype=float)
    y_pred=(p>=THRESHOLD).astype(int)

    if len(np.unique(y_true))<2:
        raise RuntimeError('TEST ground truth contains only one class; ROC/AUC is undefined.')

    metrics=competition_metrics(y_true,p,THRESHOLD)
    cm=confusion_matrix(y_true,y_pred,labels=[0,1])

    print('\n'+'='*72)
    print('OFFICIAL TEST EVALUATION — BTRICK')
    print('='*72)
    print(f"Images      : {len(y_true)}")
    print(f"Accuracy    : {metrics['Accuracy']:.6f}")
    print(f"Sensitivity : {metrics['Sensitivity']:.6f}")
    print(f"Specificity : {metrics['Specificity']:.6f}")
    print(f"F1-score    : {metrics['F1']:.6f}")
    print(f"ROC-AUC     : {metrics['AUROC']:.6f}")
    print(f"Score P     : {metrics['P']:.6f}")
    print(f"Threshold   : {metrics['threshold']:.3f}")
    print('\nConfusion Matrix [[TN, FP], [FN, TP]]:')
    print(cm)
    print(
        f"TN={metrics['TN']}  FP={metrics['FP']}  "
        f"FN={metrics['FN']}  TP={metrics['TP']}"
    )
    print('='*72)

    # Confusion Matrix figure
    plt.figure(figsize=(5,4.5))
    plt.imshow(cm)
    plt.xticks([0,1],['Normal','Pneumonia'])
    plt.yticks([0,1],['Normal','Pneumonia'])
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title('Official TEST — Confusion Matrix')
    for i in range(2):
        for j in range(2):
            plt.text(j,i,str(cm[i,j]),ha='center',va='center')
    plt.colorbar()
    plt.tight_layout()
    cm_path=OUTPUT_DIR/'Btrick_TEST_confusion_matrix.png'
    plt.savefig(cm_path,dpi=180,bbox_inches='tight')
    plt.show()

    # ROC / AUC figure
    fpr,tpr,_=roc_curve(y_true,p)
    plt.figure(figsize=(5.5,4.5))
    plt.plot(fpr,tpr,linewidth=2,label=f"AUC = {metrics['AUROC']:.6f}")
    plt.plot([0,1],[0,1],'--',linewidth=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Official TEST — ROC Curve')
    plt.grid(alpha=.25)
    plt.legend(loc='lower right')
    plt.tight_layout()
    roc_path=OUTPUT_DIR/'Btrick_TEST_roc_auc.png'
    plt.savefig(roc_path,dpi=180,bbox_inches='tight')
    plt.show()

    eval_rows=merged.copy()
    eval_rows['y_pred']=y_pred
    eval_rows['PredictedLabel']=ID_TO_LABEL[y_pred]
    eval_csv=OUTPUT_DIR/'Btrick_TEST_evaluated_predictions.csv'
    eval_rows.to_csv(eval_csv,index=False)

    eval_json=OUTPUT_DIR/'Btrick_TEST_evaluation.json'
    payload={
        'team':'Btrick',
        'n_images':int(len(y_true)),
        'threshold':float(THRESHOLD),
        'metrics':metrics,
        'confusion_matrix':cm.tolist(),
        'label_mapping':LABEL_TO_ID,
        'prediction_file':'team01.csv',
        'evaluated_predictions_file':eval_csv.name,
        'confusion_matrix_figure':cm_path.name,
        'roc_auc_figure':roc_path.name,
    }
    with open(eval_json,'w',encoding='utf-8') as f:
        json.dump(payload,f,indent=2,ensure_ascii=False)

    print('Saved evaluation JSON:',eval_json)
    print('Saved evaluated predictions:',eval_csv)
    return metrics


if official_test_dir is not None and 'submission' in globals() and 'test_audit' in globals():
    official_test_metrics=evaluate_official_test_predictions(
        official_test_dir,
        submission,
        test_audit
    )
else:
    print('Official TEST prediction has not run yet; evaluation skipped.')


## 13. Export Btrick metadata files

These two files are part of the preliminary-round package:

- `Btrick.pkl`: model information, label convention and internal metrics
- `Btrick.JSON`: internal performance metrics

They contain metadata only; the learned network itself is stored in `Btrick.keras`.


In [ ]:
MODEL_METADATA={
    'team':'Btrick',
    'task':'Normal vs Pneumonia chest X-ray classification',
    'framework':'TensorFlow/Keras final model; PyTorch training',
    'architecture':'DenseNet121 transfer-learning ensemble',
    'input_shape':[224,224,3],
    'label_mapping':LABEL_TO_ID,
    'output_order':['Normal','Pneumonia'],
    'preprocessing':[
        'grayscale',
        'replicate to 3 channels',
        'pad to square',
        'resize 224x224 bilinear',
        'ImageNet normalization',
    ],
    'augmentation_training_only':[
        'RandomAffine degrees=5',
        'translation=0.02',
        'scale=0.95-1.05',
        'brightness=0.08',
        'contrast=0.08',
    ],
    'seeds':SEEDS,
    'folds_per_seed':5,
    'ensemble_weights':SEED_WEIGHTS,
    'threshold':THRESHOLD,
    'tta':False,
    'locked_oof_metrics':LOCKED_OOF_METRICS,
    'sealed_holdout_metrics':SEALED_HOLDOUT_METRICS,
}

PKL_PATH=OUTPUT_DIR/'Btrick.pkl'
with open(PKL_PATH,'wb') as f:
    pickle.dump(MODEL_METADATA,f,pickle.HIGHEST_PROTOCOL)

PERFORMANCE_JSON={
    'team':'Btrick',
    'metric_formula':'P = 0.30*Accuracy + 0.25*Sensitivity + 0.20*Specificity + 0.25*F1',
    'label_mapping':LABEL_TO_ID,
    'threshold':THRESHOLD,
    'oof_5fold':LOCKED_OOF_METRICS,
    'sealed_holdout':SEALED_HOLDOUT_METRICS,
    'holdout_images':770,
    'holdout_retuning_performed':False,
}

JSON_PATH=OUTPUT_DIR/'Btrick.JSON'
with open(JSON_PATH,'w',encoding='utf-8') as f:
    json.dump(PERFORMANCE_JSON,f,indent=2,ensure_ascii=False)

print('Saved:',PKL_PATH)
print('Saved:',JSON_PATH)


## 14. Final submission consistency checks

Before uploading, the four core files must use the same team name and configuration:

```text
Btrick.keras
Btrick.pkl
Btrick.JSON
Btrick.ipynb
```

Consistency gates:

- input size = 224×224×3
- label convention = Normal 0, Pneumonia 1
- threshold = 0.576
- seed weights = 0.70 / 0.30
- TTA = OFF
- preprocessing in notebook/report matches TEST preprocessing
- architecture diagram matches DenseNet121 + 2-seed × 5-fold ensemble
- no hidden TEST label is read
- no TEST prediction is manually edited


In [ ]:
# Local packaging convenience: copy the parity-verified core model to its final required filename.
FINAL_KERAS=OUTPUT_DIR/'Btrick.keras'

if MODEL_PATH.resolve()!=FINAL_KERAS.resolve():
    # Only copy from a known local source; this does not modify model contents.
    shutil.copy2(MODEL_PATH,FINAL_KERAS)

print('\nFINAL PACKAGE STATUS')
print('Btrick.keras:',FINAL_KERAS.exists(), FINAL_KERAS)
print('Btrick.pkl  :',PKL_PATH.exists(), PKL_PATH)
print('Btrick.JSON :',JSON_PATH.exists(), JSON_PATH)

if FINAL_KERAS.exists():
    check_model=tf.keras.models.load_model(FINAL_KERAS,compile=False)
    assert tuple(check_model.input_shape[1:])==(224,224,3)
    assert tuple(check_model.output_shape[1:])==(2,)
    print('Final Keras load_model check: PASS')
    del check_model
    gc.collect()


## Final note

This notebook intentionally separates:

- **training-time augmentation**, used only during model development, from
- **deterministic validation/TEST preprocessing**, used before `Btrick.keras`.

This matches the organizer's clarification that each team may preprocess the downloaded TEST images before calling its submitted Keras model.


## Competition-day note

The official TEST metrics are generated only from organizer-supplied ground truth. If the downloaded package contains no labels, the notebook can still run inference and export predictions, but no valid supervised metric can be computed until labels are supplied.
